[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/information_theory/01_self_information_and_entropy/exercises.ipynb)

# Module 01 — Exercises: Self-Information and Entropy

Forty solved problems in four tiers. Every problem carries a statement, a one-line intuition, a
stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is numeric or
algorithmic — a code cell that recomputes it.

Definition, theorem and proof numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): $H$ with two random-variable arguments is joint
entropy, $D_{\mathrm{KL}}$ uses `\parallel`, and every numerical answer carries its unit — bits
for $\log_2$, nats for $\ln$.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps


def H_bits(p):
    """Shannon entropy in bits, with the convention 0 log 0 = 0."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0.0]
    return float(-(p * np.log2(p)).sum() + 0.0)


def H_nats(p):
    """Shannon entropy in nats, with the convention 0 log 0 = 0."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0.0]
    return float(-(p * np.log(p)).sum() + 0.0)


print(f"machine epsilon = {EPS:.4e}")
print("H(1/2, 1/2) =", H_bits([0.5, 0.5]), "bits")

machine epsilon = 2.2204e-16
H(1/2, 1/2) = 1.0 bits


## L0 — Concept Checks

### Problem L0.1 — Surprise of a fair coin and a loaded die

**Statement.** A fair coin lands heads. A loaded die with $\mathbb{P}(\text{six}) = 0.01$ lands
six. Give the self-information of each event in bits.

**Intuition.** Self-information depends on the probability alone, and it grows as the
probability shrinks.

**Solution.**

*Step 1.* Definition 3.1 in base two gives $I_2(x) = -\log_2 p(x)$.

*Step 2.* For the coin, $-\log_2 \tfrac{1}{2} = 1$ bit.

*Step 3.* For the die, $-\log_2 0.01 = \log_2 100 = 6.6439$ bits.

$$
\boxed{I_{\text{coin}} = 1 \text{ bit}, \qquad I_{\text{six}} = 6.6439 \text{ bits}}
$$

**Key takeaway.** Rarity is the only input to surprisal, and the scale is logarithmic.

In [2]:
print("I(heads) =", -np.log2(0.5), "bits")
print("I(six)   =", -np.log2(0.01), "bits")
assert abs(-np.log2(0.5) - 1.0) < 1e-12
assert abs(-np.log2(0.01) - 6.6439) < 5e-5

I(heads) = 1.0 bits
I(six)   = 6.643856189774724 bits


### Problem L0.2 — Why the logarithm

**Statement.** Two independent fair coins are flipped. Show that self-information is additive
over the joint outcome, and name the axiom of Theorem 4.2 this illustrates.

**Intuition.** Independence multiplies probabilities; the logarithm turns that product into a
sum.

**Solution.**

*Step 1.* The joint outcome "heads, heads" has probability
$\tfrac12 \cdot \tfrac12 = \tfrac14$.

*Step 2.* Its self-information is $-\log_2 \tfrac14 = 2$ bits, while each flip carries
$-\log_2 \tfrac12 = 1$ bit.

*Step 3.* In general $I(pq) = -\log(pq) = -\log p - \log q = I(p) + I(q)$, which is axiom 3 of
Theorem 4.2.

$$
\boxed{I(pq) = I(p) + I(q) \text{ — the additivity axiom}}
$$

**Key takeaway.** Additivity over independent events is the axiom that forces the logarithm; by
Proof 5.2 nothing else with the right monotonicity satisfies it.

In [3]:
print("I(1/4)          =", -np.log2(0.25), "bits")
print("I(1/2) + I(1/2) =", 2 * -np.log2(0.5), "bits")
assert abs(-np.log2(0.25) - 2 * -np.log2(0.5)) < 1e-12

I(1/4)          = 2.0 bits
I(1/2) + I(1/2) = 2.0 bits


### Problem L0.3 — Entropy of a deterministic variable

**Statement.** Let $\mathbb{P}(X = a) = 1$. Compute $H(X)$.

**Intuition.** There is nothing to learn from observing a variable whose value you already know.

**Solution.**

*Step 1.* Definition 3.2 sums over the support, which is the single point $a$.

*Step 2.* That one term is $-1 \cdot \log 1 = 0$; every other outcome contributes $0$ by the
convention $0 \log 0 = 0$.

$$
\boxed{H(X) = 0}
$$

**Key takeaway.** Proof 5.4 shows the converse too: $H(X) = 0$ happens only for a deterministic
$X$.

In [4]:
p = np.array([1.0, 0.0, 0.0])
print("H(X) =", H_bits(p), "bits")
assert H_bits(p) == 0.0

H(X) = 0.0 bits


### Problem L0.4 — Bits and nats

**Statement.** A distribution has entropy $H = 2.5$ bits. Express it in nats.

**Intuition.** Bits and nats are the same measurement in different units, like metres and feet.

**Solution.**

*Step 1.* $\log_2 x = \ln x / \ln 2$, so an entropy in bits becomes an entropy in nats on
multiplying by $\ln 2$.

*Step 2.* $2.5 \times 0.693147 = 1.732868$ nats.

$$
\boxed{H = 1.7329 \text{ nats}}
$$

**Key takeaway.** PyTorch, TensorFlow and JAX all use natural logarithms, so a reported
cross-entropy loss is in nats.

In [5]:
print("2.5 bits =", 2.5 * np.log(2), "nats")
assert abs(2.5 * np.log(2) - 1.7329) < 5e-5

2.5 bits = 1.7328679513998633 nats


### Problem L0.5 — Entropy of a fair die

**Statement.** Give $H(X)$ in bits for $X$ uniform on $\{1, \dots, 6\}$.

**Intuition.** The uniform distribution is the maximizer of Theorem 4.4, so its entropy is the
ceiling itself.

**Solution.**

*Step 1.* Each outcome has $p = \tfrac16$, so every term of Definition 3.2 is
$\tfrac16 \log_2 6$.

*Step 2.* Summing six identical terms gives $\log_2 6 = 2.5850$ bits.

$$
\boxed{H(X) = \log_2 6 = 2.5850 \text{ bits}}
$$

**Key takeaway.** A uniform source on $K$ symbols costs exactly $\log_2 K$ bits, and no source on
$K$ symbols costs more.

In [6]:
print("H(fair d6) =", H_bits(np.full(6, 1 / 6)), "bits   log2 6 =", np.log2(6))
assert abs(H_bits(np.full(6, 1 / 6)) - np.log2(6)) < 1e-12

H(fair d6) = 2.584962500721156 bits   log2 6 = 2.584962500721156


### Problem L0.6 — Entropy ignores the labels

**Statement.** Does $H(X)$ change if the outcomes of $X$ are relabelled or permuted?

**Intuition.** Definition 3.2 reads only the probability vector; the outcome names never enter.

**Solution.**

*Step 1.* $H(X) = -\sum_x p(x)\log p(x)$ is a sum over the support and addition is commutative.

*Step 2.* A bijective relabelling permutes the terms of that sum and leaves the total unchanged.

$$
\boxed{H \text{ is invariant under any bijective relabelling of the outcomes}}
$$

**Key takeaway.** This is why entropy is the right uncertainty measure for categorical labels,
tokens and symbols, where the values carry no order or magnitude.

In [7]:
p = np.array([0.5, 0.2, 0.2, 0.1])
perm = rng.permutation(4)
print("H(p)        =", H_bits(p), "bits")
print("H(permuted) =", H_bits(p[perm]), "bits   permutation:", perm)
assert H_bits(p) == H_bits(p[perm])

H(p)        = 1.7609640474436812 bits
H(permuted) = 1.7609640474436812 bits   permutation: [2 0 1 3]


### Problem L0.7 — The convention $0 \log 0 = 0$

**Statement.** Justify the convention $0 \log 0 = 0$ used in Definition 3.2.

**Intuition.** A probability going to zero kills its term faster than the logarithm blows it up.

**Solution.**

*Step 1.* Write $t \log_2 t$ and let $t \to 0^{+}$.

*Step 2.* Substituting $t = 2^{-s}$ gives $-s\,2^{-s}$, and the exponential beats the linear
factor, so the limit is $0$.

$$
\boxed{\lim_{t \to 0^{+}} t \log t = 0, \text{ so impossible outcomes contribute nothing}}
$$

**Key takeaway.** In code the convention must be applied explicitly: `0 * np.log(0)` is `nan`,
not `0`.

In [8]:
for t in (1e-3, 1e-6, 1e-12, 1e-100):
    print(f"  t = {t:.0e}   t log2 t = {t * np.log2(t):+.4e}")
with np.errstate(divide="ignore", invalid="ignore"):
    naive = 0.0 * np.log(0.0)
print("naive 0 * log(0) in IEEE arithmetic:", naive)
assert np.isnan(naive)
assert abs(1e-12 * np.log2(1e-12)) < 1e-10

  t = 1e-03   t log2 t = -9.9658e-03
  t = 1e-06   t log2 t = -1.9932e-05
  t = 1e-12   t log2 t = -3.9863e-11
  t = 1e-100   t log2 t = -3.3219e-98
naive 0 * log(0) in IEEE arithmetic: nan


### Problem L0.8 — Reading a Kraft sum

**Statement.** Which of the binary length profiles $(1,2,3,3)$ and $(1,1,2)$ can be the
codeword lengths of a prefix code?

**Intuition.** Theorem 4.8 turns the question into one arithmetic sum.

**Solution.**

*Step 1.* For $(1,2,3,3)$ the Kraft sum is
$\tfrac12 + \tfrac14 + \tfrac18 + \tfrac18 = 1 \le 1$, so a prefix code exists.

*Step 2.* For $(1,1,2)$ it is $\tfrac12 + \tfrac12 + \tfrac14 = 1.25 \gt 1$, so none does.

$$
\boxed{(1,2,3,3) \text{ is feasible};\ (1,1,2) \text{ is not}}
$$

**Key takeaway.** The two codewords of length $1$ already exhaust the tree, leaving nothing for a
third codeword.

In [9]:
for lengths in ([1, 2, 3, 3], [1, 1, 2]):
    ksum = float((2.0 ** -np.array(lengths)).sum())
    print(f"lengths {lengths}: Kraft sum = {ksum:.5f}   feasible: {ksum <= 1.0}")
assert float((2.0 ** -np.array([1, 2, 3, 3])).sum()) == 1.0
assert float((2.0 ** -np.array([1, 1, 2])).sum()) > 1.0

lengths [1, 2, 3, 3]: Kraft sum = 1.00000   feasible: True
lengths [1, 1, 2]: Kraft sum = 1.25000   feasible: False


## L1 — Foundations

### Problem L1.1 — Entropy of a dyadic distribution

**Statement.** Compute the entropy in bits of
$p = \left(\tfrac12, \tfrac14, \tfrac18, \tfrac18\right)$, and compare it to the uniform
ceiling.

**Intuition.** Every probability is a power of two, so every self-information is an integer.

**Solution.**

*Step 1.* The four terms of Definition 3.2 are

$$
\tfrac12 \log_2 2, \quad \tfrac14 \log_2 4, \quad \tfrac18 \log_2 8, \quad \tfrac18 \log_2 8 .
$$

*Step 2.* Evaluating, $0.5 + 0.5 + 0.375 + 0.375 = 1.75$ bits.

*Step 3.* Theorem 4.4 gives the ceiling $\log_2 4 = 2$ bits, so this source is $0.25$ bits
cheaper than the worst four-symbol source.

$$
\boxed{H(p) = 1.75 \text{ bits}}
$$

**Key takeaway.** For dyadic distributions the entropy is attained exactly by an integer-length
prefix code, which is the equality case of Theorem 4.9.

In [10]:
p = np.array([0.5, 0.25, 0.125, 0.125])
ell = np.ceil(-np.log2(p)).astype(int)
print("H(p)        =", H_bits(p), "bits")
print("ceiling     =", np.log2(4), "bits")
print("Shannon E[L]=", float((p * ell).sum()), "bits with lengths", ell)
assert H_bits(p) == 1.75
assert float((p * ell).sum()) == H_bits(p)

H(p)        = 1.75 bits
ceiling     = 2.0 bits
Shannon E[L]= 1.75 bits with lengths [1 2 3 3]


### Problem L1.2 — Maximum of the binary entropy function

**Statement.** Show by calculus that $H_b(p)$ is maximized at $p = \tfrac12$ and give the
maximum.

**Intuition.** The curve of Section 2 of the theory notebook is concave with a single interior
critical point.

**Solution.**

*Step 1.* Work in nats: $H(p) = -p\ln p - (1-p)\ln(1-p)$. Differentiating,

$$
H'(p) = -\ln p - 1 + \ln(1-p) + 1 = \ln\frac{1-p}{p}.
$$

*Step 2.* Setting $H'(p) = 0$ gives $\tfrac{1-p}{p} = 1$, so $p = \tfrac12$.

*Step 3.* The second derivative is

$$
H''(p) = -\frac{1}{p} - \frac{1}{1-p} \lt 0 \quad \text{on } (0,1),
$$

so $H$ is strictly concave and the critical point is the unique global maximum.

*Step 4.* Its value is $H_b(\tfrac12) = 1$ bit.

$$
\boxed{p^{\star} = \tfrac12, \qquad H_b(p^{\star}) = 1 \text{ bit}}
$$

**Key takeaway.** A fair coin is the least predictable binary source, and concavity is what
turns a critical point into a global maximum.

In [11]:
grid = np.linspace(1e-9, 1 - 1e-9, 200001)
Hb = -grid * np.log2(grid) - (1 - grid) * np.log2(1 - grid)
imax = int(np.argmax(Hb))
print("argmax p =", grid[imax], "   max H_b =", Hb[imax], "bits")
assert abs(grid[imax] - 0.5) < 1e-5
assert abs(Hb[imax] - 1.0) < 1e-9

argmax p = 0.5000000000000001    max H_b = 1.0 bits


### Problem L1.3 — Entropy of the geometric distribution

**Statement.** Let $\mathbb{P}(X = k) = (1-q) q^{k}$ for $k = 0, 1, 2, \dots$ with
$0 \lt q \lt 1$. Compute $H(X)$ in nats.

**Intuition.** The log-pmf is affine in $k$, so the entropy reduces to the mean.

**Solution.**

*Step 1.* $\ln p(k) = \ln(1-q) + k \ln q$, so

$$
H(X) = -\sum_{k \ge 0} p(k)\left[\ln(1-q) + k \ln q\right] = -\ln(1-q) - \mathbb{E}[X]\ln q .
$$

*Step 2.* Differentiating $\sum_k q^k = (1-q)^{-1}$ gives
$\mathbb{E}[X] = \tfrac{q}{1-q}$.

*Step 3.* Substituting,

$$
H(X) = -\ln(1-q) - \frac{q \ln q}{1-q} = \frac{-(1-q)\ln(1-q) - q\ln q}{1-q} = \frac{H_b(q)}{1-q},
$$

with $H_b$ measured in nats.

$$
\boxed{H(X) = \frac{H_b(q)}{1-q} \text{ nats}}
$$

**Key takeaway.** Per-trial uncertainty $H_b(q)$ amplified by the expected number of trials
$1/(1-q)$; problem L3.3 shows this is the maximum-entropy law at fixed mean.

In [12]:
for q in (0.3, 0.5, 0.8):
    k = np.arange(0, 20000)
    pk = (1 - q) * q ** k
    closed = H_nats([q, 1 - q]) / (1 - q)
    print(f"q = {q}:  direct = {H_nats(pk):.12f}   closed form = {closed:.12f} nats")
    assert abs(H_nats(pk) - closed) < 1e-9

q = 0.3:  direct = 0.872663288650   closed form = 0.872663288650 nats
q = 0.5:  direct = 1.386294361120   closed form = 1.386294361120 nats
q = 0.8:  direct = 2.502012117691   closed form = 2.502012117691 nats


### Problem L1.4 — Entropy under grouping

**Statement.** For $p = \left(\tfrac12, \tfrac13, \tfrac16\right)$ verify the grouping identity
$H(p) = H_b(\tfrac12) + \tfrac12 H\!\left(\tfrac23, \tfrac13\right)$.

**Intuition.** Choosing in two stages costs the same as choosing in one, weighted by the chance
of reaching the second stage.

**Solution.**

*Step 1.* Directly,

$$
H(p) = \tfrac12 \log_2 2 + \tfrac13 \log_2 3 + \tfrac16 \log_2 6 = 0.5 + 0.5283 + 0.4308 = 1.4591 \text{ bits}.
$$

*Step 2.* Stage one separates outcome $1$ from the block $\{2,3\}$, both of probability
$\tfrac12$, costing $H_b(\tfrac12) = 1$ bit.

*Step 3.* Stage two is reached with probability $\tfrac12$ and splits the block with conditional
probabilities $(\tfrac23, \tfrac13)$, costing

$$
H\!\left(\tfrac23, \tfrac13\right) = \tfrac23\log_2\tfrac32 + \tfrac13\log_2 3 = 0.9183 \text{ bits}.
$$

*Step 4.* Total: $1 + \tfrac12(0.9183) = 1.4591$ bits, matching Step 1.

$$
\boxed{H(p) = 1 + \tfrac12 H\!\left(\tfrac23, \tfrac13\right) = 1.4591 \text{ bits}}
$$

**Key takeaway.** This is axiom 4 of Theorem 4.3. Problem L3.1 shows that this one identity,
plus continuity and monotonicity, pins entropy down completely.

In [13]:
p = np.array([1 / 2, 1 / 3, 1 / 6])
w = p[1] + p[2]
two_stage = H_bits([0.5, 0.5]) + w * H_bits([p[1] / w, p[2] / w])
print("direct    =", H_bits(p), "bits")
print("two-stage =", two_stage, "bits")
print(f"residual  = {abs(H_bits(p) - two_stage):.3e}")
assert abs(H_bits(p) - two_stage) < 1e-12

direct    = 1.4591479170272448 bits
two-stage = 1.4591479170272448 bits
residual  = 0.000e+00


### Problem L1.5 — Differential entropy of the uniform distribution

**Statement.** Compute $h(X)$ for $X \sim \mathrm{Uniform}(0, a)$ with $a \gt 0$, and say for
which $a$ it is negative.

**Intuition.** A constant density makes the integral of Definition 3.4 collapse to one term.

**Solution.**

*Step 1.* The density is $f(x) = 1/a$ on $(0,a)$, so

$$
h(X) = -\int_0^{a} \frac{1}{a}\ln\frac{1}{a}\,dx = -\ln\frac{1}{a} = \ln a \text{ nats}.
$$

*Step 2.* $\ln a \lt 0$ exactly when $a \lt 1$.

$$
\boxed{h(X) = \ln a \text{ nats}, \qquad h \lt 0 \iff a \lt 1}
$$

**Key takeaway.** Differential entropy is not a bit count. It is a density-dependent quantity
that shifts under rescaling, so only differences of differential entropies carry absolute
meaning.

In [14]:
for a in (0.5, 1.0, 2.0, np.e):
    print(f"a = {a:.4f}:  h = ln a = {np.log(a):+.6f} nats")
assert np.log(0.5) < 0 < np.log(2.0)

a = 0.5000:  h = ln a = -0.693147 nats
a = 1.0000:  h = ln a = +0.000000 nats
a = 2.0000:  h = ln a = +0.693147 nats
a = 2.7183:  h = ln a = +1.000000 nats


### Problem L1.6 — Differential entropy of a Gaussian

**Statement.** Derive $h(X) = \tfrac12 \ln(2\pi e \sigma^2)$ for
$X \sim \mathcal{N}(\mu, \sigma^2)$ and evaluate it at $\sigma = 1$.

**Intuition.** The log-density is a quadratic, so its expectation needs only the variance.

**Solution.**

*Step 1.* With $f$ the normal density,

$$
-\ln f(x) = \tfrac12 \ln(2\pi\sigma^2) + \frac{(x-\mu)^2}{2\sigma^2}.
$$

*Step 2.* Taking the expectation and using $\mathbb{E}[(X-\mu)^2] = \sigma^2$,

$$
h(X) = \tfrac12\ln(2\pi\sigma^2) + \tfrac12 = \tfrac12\ln\left(2\pi e\sigma^2\right).
$$

*Step 3.* At $\sigma = 1$ this is $\tfrac12\ln(2\pi e) = 1.4189$ nats, that is $2.0471$ bits.

$$
\boxed{h(X) = \tfrac12\ln(2\pi e \sigma^2) = 1.4189 \text{ nats at } \sigma = 1}
$$

**Key takeaway.** Doubling $\sigma$ adds exactly $\ln 2$ nats, one bit, which is Theorem 4.7's
bound moving with the power budget.

In [15]:
from scipy.integrate import quad

for s in (1.0, 2.0):
    def integrand(x, s=s):
        f = np.exp(-x * x / (2 * s * s)) / np.sqrt(2 * np.pi * s * s)
        return f * (0.5 * np.log(2 * np.pi * s * s) + x * x / (2 * s * s))
    num, _ = quad(integrand, -12 * s, 12 * s, limit=400)
    closed = 0.5 * np.log(2 * np.pi * np.e * s * s)
    print(f"sigma = {s}: quad = {num:.12f}   closed = {closed:.12f} nats"
          f"   = {closed / np.log(2):.6f} bits")
    assert abs(num - closed) < 1e-10
print("h(sigma=2) - h(sigma=1) =", 0.5 * np.log(2 * np.pi * np.e * 4)
      - 0.5 * np.log(2 * np.pi * np.e), " ln 2 =", np.log(2))

sigma = 1.0: quad = 1.418938533205   closed = 1.418938533205 nats   = 2.047096 bits
sigma = 2.0: quad = 2.112085713765   closed = 2.112085713765 nats   = 3.047096 bits
h(sigma=2) - h(sigma=1) = 0.6931471805599454  ln 2 = 0.6931471805599453


### Problem L1.7 — Support size only bounds entropy

**Statement.** Build a distribution on $100$ outcomes whose entropy is far below the ceiling
$\log_2 100$, and quantify the gap.

**Intuition.** Theorem 4.4 bounds entropy by the support size but says nothing from below beyond
zero.

**Solution.**

*Step 1.* Take $p_1 = 0.9$ and $p_2 = \dots = p_{100} = 0.1/99$.

*Step 2.* The dominant term contributes $-0.9\log_2 0.9 = 0.1368$ bits.

*Step 3.* Each of the $99$ small terms contributes
$-\tfrac{0.1}{99}\log_2 \tfrac{0.1}{99} = 0.010052$ bits, totalling $0.9951$ bits.

*Step 4.* Hence $H(p) = 1.1319$ bits against a ceiling of $\log_2 100 = 6.6439$ bits.

$$
\boxed{H(p) = 1.1319 \text{ bits} \; \ll \; \log_2 100 = 6.6439 \text{ bits}}
$$

**Key takeaway.** Counting outcomes is not measuring uncertainty; only the shape of the mass is.

In [16]:
p = np.concatenate([[0.9], np.full(99, 0.1 / 99)])
print("mass check :", p.sum())
print("H(p)       =", H_bits(p), "bits")
print("ceiling    =", np.log2(100), "bits")
print("first term =", -0.9 * np.log2(0.9), "bits")
assert abs(H_bits(p) - 1.1319) < 5e-5
assert H_bits(p) < np.log2(100)

mass check : 1.0
H(p)       = 1.1319312555972418 bits
ceiling    = 6.643856189774724 bits
first term = 0.13680278410054494 bits


### Problem L1.8 — Entropy of a deterministic function

**Statement.** Let $X$ be uniform on $\{0,1,2,3\}$ and $g(x) = x \bmod 2$. Compare $H(g(X))$
with $H(X)$ and state when equality can hold.

**Intuition.** A two-to-one map pools outcomes, and pooling can only destroy distinctions.

**Solution.**

*Step 1.* $H(X) = \log_2 4 = 2$ bits.

*Step 2.* $g(X)$ is a fair coin, so $H(g(X)) = 1$ bit.

*Step 3.* Theorem 4.6 predicts $H(g(X)) \le H(X)$ with equality only for $g$ injective on the
support; $g$ here is two-to-one, so the inequality is strict.

$$
\boxed{H(g(X)) = 1 \text{ bit} \lt 2 \text{ bits} = H(X)}
$$

**Key takeaway.** Deterministic post-processing never creates information. A randomized map can:
adding an independent fair coin to $X$ raises the entropy by exactly one bit.

In [17]:
pX = np.full(4, 0.25)
pY = np.array([pX[0] + pX[2], pX[1] + pX[3]])
print("H(X)    =", H_bits(pX), "bits")
print("H(g(X)) =", H_bits(pY), "bits")
p_joint = np.kron(pX, np.array([0.5, 0.5]))
print("H(X, fair coin) =", H_bits(p_joint), "bits  (randomization adds one bit)")
assert H_bits(pY) < H_bits(pX)
assert abs(H_bits(p_joint) - (H_bits(pX) + 1)) < 1e-12

H(X)    = 2.0 bits
H(g(X)) = 1.0 bits
H(X, fair coin) = 3.0 bits  (randomization adds one bit)


### Problem L1.9 — Shannon code lengths and the one-bit gap

**Statement.** For $p = (0.5, 0.3, 0.2)$ compute $H(p)$, the Shannon lengths
$\ell(x) = \lceil \log_2(1/p(x))\rceil$, the Kraft sum, and $\mathbb{E}[L]$. Check the bracket of
Theorem 4.9.

**Intuition.** Rounding the ideal lengths up costs less than one bit per symbol.

**Solution.**

*Step 1.* $H(p) = -0.5\log_2 0.5 - 0.3\log_2 0.3 - 0.2\log_2 0.2 = 1.4855$ bits.

*Step 2.* The ideal lengths $-\log_2 p$ are $(1, 1.7370, 2.3219)$, so
$\ell = (1, 2, 3)$.

*Step 3.* The Kraft sum is $\tfrac12 + \tfrac14 + \tfrac18 = 0.875 \le 1$, so by Theorem 4.8 a
prefix code with these lengths exists.

*Step 4.* $\mathbb{E}[L] = 0.5(1) + 0.3(2) + 0.2(3) = 1.7$ bits, which lies in
$[1.4855, 2.4855)$ with a gap of $0.2145$ bits.

$$
\boxed{H = 1.4855,\ \mathbb{E}[L] = 1.7 \text{ bits, gap } 0.2145 \lt 1}
$$

**Key takeaway.** The gap is pure integer rounding; Section 7.3 of the theory notebook shows it
falling like $1/n$ under block coding.

In [18]:
p = np.array([0.5, 0.3, 0.2])
ell = np.ceil(-np.log2(p)).astype(int)
EL = float((p * ell).sum())
print("H(p)      =", H_bits(p), "bits")
print("ideal     =", -np.log2(p))
print("lengths   =", ell, "  Kraft sum =", float((2.0 ** -ell).sum()))
print("E[L]      =", EL, "bits   gap =", EL - H_bits(p))
assert float((2.0 ** -ell).sum()) <= 1.0
assert H_bits(p) <= EL < H_bits(p) + 1

H(p)      = 1.4854752972273344 bits
ideal     = [1.     1.737  2.3219]
lengths   = [1 2 3]   Kraft sum = 0.875
E[L]      = 1.7000000000000002 bits   gap = 0.21452470277266578


### Problem L1.10 — Concavity on an explicit mixture

**Statement.** With $p = (0.7, 0.2, 0.1)$, $q = (0.1, 0.2, 0.7)$ and $\lambda = 0.25$, compare
$H(\lambda p + (1-\lambda)q)$ with $\lambda H(p) + (1-\lambda)H(q)$.

**Intuition.** Blending two peaked distributions spreads the mass out.

**Solution.**

*Step 1.* $p$ and $q$ are permutations of each other, so $H(p) = H(q) = 1.1568$ bits and the
right-hand side equals $1.1568$ bits for every $\lambda$.

*Step 2.* The mixture is
$0.25 p + 0.75 q = (0.25, 0.20, 0.55)$.

*Step 3.* Its entropy is $1.4388$ bits, which exceeds $1.1568$ bits.

$$
\boxed{H(\lambda p + (1-\lambda) q) = 1.4388 \gt 1.1568 = \lambda H(p) + (1-\lambda) H(q)}
$$

**Key takeaway.** Theorem 4.5 with $p \neq q$, hence strict. The same concavity makes the
plug-in entropy estimator biased low, as measured in Section 7.6 of the theory notebook.

In [19]:
p = np.array([0.7, 0.2, 0.1])
q = np.array([0.1, 0.2, 0.7])
lam = 0.25
mix = lam * p + (1 - lam) * q
print("H(p), H(q)   =", H_bits(p), H_bits(q), "bits")
print("mixture      =", mix)
print("H(mixture)   =", H_bits(mix), "bits")
print("lam-average  =", lam * H_bits(p) + (1 - lam) * H_bits(q), "bits")
assert H_bits(mix) > lam * H_bits(p) + (1 - lam) * H_bits(q)

H(p), H(q)   = 1.1567796494470395 1.1567796494470395 bits
mixture      = [0.25 0.2  0.55]
H(mixture)   = 1.4387586809150084 bits
lam-average  = 1.1567796494470395 bits


### Problem L1.11 — Differential entropy under scaling

**Statement.** Show $h(aX) = h(X) + \ln \lvert a \rvert$ for $a \neq 0$, and check it on a
Gaussian.

**Intuition.** Stretching the axis by $a$ divides the density by $\lvert a \rvert$, and the
logarithm turns that into an additive shift.

**Solution.**

*Step 1.* If $Y = aX$ then $f_Y(y) = f_X(y/a)/\lvert a \rvert$.

*Step 2.* Substituting into Definition 3.4 with $x = y/a$, so $dy = \lvert a \rvert\,dx$,

$$
h(Y) = -\int \frac{f_X(x)}{\lvert a \rvert}\ln\frac{f_X(x)}{\lvert a \rvert}\, \lvert a \rvert \, dx
= -\int f_X \ln f_X \, dx + \ln \lvert a \rvert .
$$

*Step 3.* For $X \sim \mathcal{N}(0,1)$ and $a = 3$ this predicts
$1.4189 + \ln 3 = 2.5176$ nats, which is $\tfrac12\ln(2\pi e \cdot 9)$.

$$
\boxed{h(aX) = h(X) + \ln \lvert a \rvert}
$$

**Key takeaway.** Differential entropy is not invariant under a change of units, which is
exactly why only differences of it — divergences and mutual information — are physically
meaningful.

In [20]:
h1 = 0.5 * np.log(2 * np.pi * np.e)
for a in (0.5, 3.0):
    predicted = h1 + np.log(abs(a))
    direct = 0.5 * np.log(2 * np.pi * np.e * a * a)
    print(f"a = {a}:  h(X) + ln|a| = {predicted:.12f}   h(N(0,a^2)) = {direct:.12f} nats")
    assert abs(predicted - direct) < 1e-12

a = 0.5:  h(X) + ln|a| = 0.725791352645   h(N(0,a^2)) = 0.725791352645 nats
a = 3.0:  h(X) + ln|a| = 2.517550821873   h(N(0,a^2)) = 2.517550821873 nats


### Problem L1.12 — Renyi entropy and the limit $\alpha \to 1$

**Statement.** The Renyi entropy of order $\alpha \neq 1$ is

$$
H_{\alpha}(p) = \frac{1}{1-\alpha}\log_2 \sum_k p_k^{\alpha} .
$$

Show $H_{\alpha} \to H$ as $\alpha \to 1$, and evaluate the family at $p = (0.7, 0.2, 0.1)$.

**Intuition.** Both numerator and denominator vanish at $\alpha = 1$; one application of
l'Hopital's rule recovers Shannon.

**Solution.**

*Step 1.* Let $g(\alpha) = \ln \sum_k p_k^{\alpha}$. Then $g(1) = \ln 1 = 0$, so
$H_{\alpha} = g(\alpha) / ((1-\alpha)\ln 2)$ is a $0/0$ form at $\alpha = 1$.

*Step 2.* Differentiate: $g'(\alpha) = \dfrac{\sum_k p_k^{\alpha}\ln p_k}{\sum_k p_k^{\alpha}}$,
so $g'(1) = \sum_k p_k \ln p_k$.

*Step 3.* By l'Hopital,

$$
\lim_{\alpha \to 1} H_{\alpha} = \frac{g'(1)}{-\ln 2} = -\frac{1}{\ln 2}\sum_k p_k \ln p_k = H(p) \text{ bits}.
$$

*Step 4.* Numerically at $p = (0.7, 0.2, 0.1)$: $H_{0.5} = 1.3563$, $H_1 = 1.1568$,
$H_2 = 0.8890$ and $H_{\infty} = -\log_2 0.7 = 0.5146$ bits.

$$
\boxed{H_{\alpha} \to H \text{ as } \alpha \to 1, \text{ and } H_{\alpha} \text{ decreases in } \alpha}
$$

**Key takeaway.** Shannon entropy is the $\alpha = 1$ member of a decreasing family; the
$\alpha = 2$ member is the collision entropy used in randomness extraction, and
$\alpha = \infty$ is the min-entropy used in cryptography.

In [21]:
p = np.array([0.7, 0.2, 0.1])


def renyi(p, alpha):
    return float(np.log2((p ** alpha).sum()) / (1 - alpha))


for a in (0.5, 0.9, 0.99, 0.999, 1.001, 2.0, 3.0):
    print(f"  alpha = {a:<6}  H_alpha = {renyi(p, a):.6f} bits")
print("  alpha -> 1 : Shannon  =", H_bits(p), "bits")
print("  alpha = inf: min-ent  =", -np.log2(p.max()), "bits")
assert abs(renyi(p, 1.000001) - H_bits(p)) < 1e-5
assert renyi(p, 0.5) > H_bits(p) > renyi(p, 2.0) > -np.log2(p.max())

  alpha = 0.5     H_alpha = 1.356327 bits
  alpha = 0.9     H_alpha = 1.193352 bits
  alpha = 0.99    H_alpha = 1.160355 bits
  alpha = 0.999   H_alpha = 1.157136 bits
  alpha = 1.001   H_alpha = 1.156423 bits
  alpha = 2.0     H_alpha = 0.888969 bits
  alpha = 3.0     H_alpha = 0.753176 bits
  alpha -> 1 : Shannon  = 1.1567796494470395 bits
  alpha = inf: min-ent  = 0.5145731728297583 bits


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Information gain of a decision-tree split

**Statement.** A node holds $8$ positive and $8$ negative examples. A feature splits it into a
left child with $(6, 2)$ and a right child with $(2, 6)$. Compute the information gain in bits.

**Intuition.** Information gain is the entropy the split removes, averaged over the children by
their sizes.

**Solution.**

*Step 1.* The parent classes are balanced, so $H_{\text{parent}} = H_b(\tfrac12) = 1$ bit.

*Step 2.* Each child has class proportions $(\tfrac34, \tfrac14)$, so

$$
H_b(\tfrac14) = -\tfrac14\log_2\tfrac14 - \tfrac34\log_2\tfrac34 = 0.5 + 0.3113 = 0.8113 \text{ bits}.
$$

*Step 3.* Both children hold half the data, so the expected child entropy is also $0.8113$ bits.

*Step 4.* The gain is $1 - 0.8113 = 0.1887$ bits.

$$
\boxed{\mathrm{IG} = 0.1887 \text{ bits}}
$$

**Key takeaway.** ID3 and C4.5 greedily maximize exactly this quantity. It is non-negative for
every split, by the concavity of Theorem 4.5 — which is precisely why an unpruned tree keeps
splitting until the leaves are pure.

In [22]:
parent = np.array([8, 8]) / 16
left = np.array([6, 2]) / 8
right = np.array([2, 6]) / 8
IG = H_bits(parent) - 0.5 * H_bits(left) - 0.5 * H_bits(right)
print("H(parent) =", H_bits(parent), "bits")
print("H(child)  =", H_bits(left), "bits")
print("gain      =", IG, "bits")
assert abs(IG - 0.1887) < 5e-5
assert IG >= 0.0

H(parent) = 1.0 bits
H(child)  = 0.8112781244591328 bits
gain      = 0.1887218755408671 bits


### Problem L2.2 — Perplexity of a language model

**Statement.** A model achieves an average cross-entropy of $1.2$ nats per token on held-out
text. Give its perplexity and its bits per token.

**Intuition.** Perplexity is the effective number of equally likely choices per step.

**Solution.**

*Step 1.* Perplexity is the exponential of the per-token cross-entropy:
$\mathrm{PPL} = e^{1.2} = 3.3201$.

*Step 2.* In bits, $1.2 / \ln 2 = 1.7312$ bits per token.

*Step 3.* Consistently, $2^{1.7312} = 3.3201$: the two units name the same branching factor.

$$
\boxed{\mathrm{PPL} = 3.3201, \qquad 1.7312 \text{ bits per token}}
$$

**Key takeaway.** By Theorem 4.9 that score is a file size: bits per token is exactly the rate at
which the model would compress the corpus, so language modelling and compression are one problem.

In [23]:
ce_nats = 1.2
print("perplexity    =", np.exp(ce_nats))
print("bits per token=", ce_nats / np.log(2))
print("2 ** bits     =", 2 ** (ce_nats / np.log(2)))
assert abs(np.exp(ce_nats) - 3.3201) < 5e-5
assert abs(2 ** (ce_nats / np.log(2)) - np.exp(ce_nats)) < 1e-12

perplexity    = 3.3201169227365472
bits per token= 1.7312340490667562
2 ** bits     = 3.3201169227365477


### Problem L2.3 — Entropy of a softmax and the temperature dial

**Statement.** Logits $z = (2, 0, 0)$ feed a temperature-scaled softmax $p_k \propto e^{z_k/T}$.
Compute $H(p)$ in nats at $T = 1$, and describe the limits $T \to 0^{+}$ and $T \to \infty$.

**Intuition.** Temperature interpolates between the argmax and the uniform distribution, so it
interpolates between zero entropy and the ceiling.

**Solution.**

*Step 1.* At $T = 1$ the unnormalized weights are $(e^2, 1, 1) = (7.3891, 1, 1)$ with normalizer
$Z = 9.3891$, giving $p = (0.7870, 0.1065, 0.1065)$.

*Step 2.* Definition 3.2 in nats gives

$$
H(p) = -0.7870\ln 0.7870 - 2(0.1065)\ln 0.1065 = 0.1885 + 0.4771 = 0.6656 \text{ nats}.
$$

*Step 3.* As $T \to 0^{+}$ the largest logit dominates and $p \to (1,0,0)$, so $H \to 0$, the
deterministic case of Theorem 4.4.

*Step 4.* As $T \to \infty$ the logits flatten and $p \to (\tfrac13,\tfrac13,\tfrac13)$, so
$H \to \ln 3 = 1.0986$ nats, the uniform ceiling of Theorem 4.4.

$$
\boxed{H(T{=}1) = 0.6656 \text{ nats}; \quad H \to 0 \text{ as } T \to 0^{+}, \quad H \to \ln 3 \text{ as } T \to \infty}
$$

**Key takeaway.** Sampling temperature is an entropy dial, and Theorem 4.4 supplies both of its
end stops.

In [24]:
from scipy.special import logsumexp

z = np.array([2.0, 0.0, 0.0])
for T in (0.05, 0.5, 1.0, 10.0, 1000.0):
    p = np.exp(z / T - logsumexp(z / T))
    print(f"T = {T:>7}:  p = {np.round(p, 4)}   H = {H_nats(p):.6f} nats")
p1 = np.exp(z - logsumexp(z))
assert abs(H_nats(p1) - 0.6656) < 5e-5
assert H_nats(np.exp(z / 1000 - logsumexp(z / 1000))) < np.log(3) + 1e-12

T =    0.05:  p = [1. 0. 0.]   H = 0.000000 nats
T =     0.5:  p = [0.9647 0.0177 0.0177]   H = 0.177324 nats
T =     1.0:  p = [0.787  0.1065 0.1065]   H = 0.665573 nats
T =    10.0:  p = [0.3792 0.3104 0.3104]   H = 1.093986 nats
T =  1000.0:  p = [0.3338 0.3331 0.3331]   H = 1.098612 nats


### Problem L2.4 — Shannon's guessing game and the entropy of English

**Statement.** In Shannon's 1951 experiment a subject guesses the next letter of English text.
Explain why any such guesser upper-bounds the entropy rate, and quantify the redundancy if an
optimal guesser achieves $1.3$ bits per letter against an alphabet of $27$ symbols.

**Intuition.** A predictor is a code, and any code's average length is at least the entropy —
so any predictor's score is an upper bound on it.

**Solution.**

*Step 1.* A guesser defines a predictive distribution $q$ over the next letter. Coding the text
with $q$ costs $H_{\times}(p, q)$ bits per letter on average.

*Step 2.* By Theorem 4.9 no code beats $H(p)$, so $H(p) \le H_{\times}(p, q)$ for every $q$: the
guesser's score is an upper bound, and better guessers tighten it.

*Step 3.* With $1.3$ bits per letter achieved and a uniform ceiling of
$\log_2 27 = 4.7549$ bits, the redundancy is

$$
1 - \frac{1.3}{4.7549} = 0.7266 .
$$

$$
\boxed{H(\text{English}) \le 1.3 \text{ bits per letter}, \text{ redundancy } 72.7\%}
$$

**Key takeaway.** Evaluating a modern language model in bits per character is Shannon's 1951
experiment with the human guesser replaced by a network; the bound direction is unchanged.

In [25]:
ceiling = np.log2(27)
achieved = 1.3
print("uniform ceiling log2 27 =", ceiling, "bits per letter")
print("achieved                =", achieved, "bits per letter")
print("redundancy              =", 1 - achieved / ceiling)
assert abs(1 - achieved / ceiling - 0.7266) < 5e-5

uniform ceiling log2 27 = 4.754887502163468 bits per letter
achieved                = 1.3 bits per letter
redundancy              = 0.7265971067857018


### Problem L2.5 — Entropy regularization in reinforcement learning

**Statement.** A two-action agent maximizes $J(\pi) = \mathbb{E}_{a \sim \pi}[Q(a)] + \alpha H(\pi)$
with $Q = (1, 0)$ and $\alpha \gt 0$, entropy in nats. Show the optimum is
$\pi^{\star}(a) \propto e^{Q(a)/\alpha}$ and evaluate it at $\alpha = 0.5$.

**Intuition.** An entropy bonus pays the agent to stay uncertain, and the trade-off against
reward is exactly a Boltzmann distribution.

**Solution.**

*Step 1.* Write $\pi = (\theta, 1-\theta)$, so

$$
J(\theta) = \theta + \alpha\left[-\theta\ln\theta - (1-\theta)\ln(1-\theta)\right].
$$

*Step 2.* Differentiating,

$$
J'(\theta) = 1 + \alpha \ln\frac{1-\theta}{\theta}.
$$

*Step 3.* Setting $J'(\theta) = 0$ gives $\theta / (1-\theta) = e^{1/\alpha}$, that is
$\theta = e^{1/\alpha} / (1 + e^{1/\alpha})$, which is the softmax of $Q/\alpha$.

*Step 4.* $J$ is a sum of a linear function and a strictly concave one, so this stationary point
is the unique maximum. At $\alpha = 0.5$, $e^{2} = 7.3891$ and $\theta = 0.8808$.

$$
\boxed{\pi^{\star} = (0.8808,\ 0.1192) \text{ at } \alpha = 0.5}
$$

**Key takeaway.** This is the maximum-entropy computation of problem L3.8 with $Q$ in place of
energy and $\alpha$ in place of temperature: soft actor-critic is statistical mechanics with
rewards.

In [26]:
Q = np.array([1.0, 0.0])
for alpha in (0.1, 0.5, 2.0):
    pi = np.exp(Q / alpha - logsumexp(Q / alpha))
    grid = np.linspace(1e-9, 1 - 1e-9, 200001)
    J = grid * 1.0 + alpha * (-grid * np.log(grid) - (1 - grid) * np.log(1 - grid))
    print(f"alpha = {alpha}:  closed form = {np.round(pi, 6)}   "
          f"grid argmax theta = {grid[int(np.argmax(J))]:.6f}")
    assert abs(pi[0] - grid[int(np.argmax(J))]) < 1e-5
pi_half = np.exp(Q / 0.5 - logsumexp(Q / 0.5))
assert abs(pi_half[0] - 0.8808) < 5e-5

alpha = 0.1:  closed form = [1. 0.]   grid argmax theta = 0.999955
alpha = 0.5:  closed form = [0.8808 0.1192]   grid argmax theta = 0.880795
alpha = 2.0:  closed form = [0.6225 0.3775]   grid argmax theta = 0.622460


### Problem L2.6 — Bias of the plug-in entropy estimator

**Statement.** Explain why $\hat{H} = -\sum_k \hat{p}_k \ln \hat{p}_k$ from $n$ samples
underestimates $H(p)$, and state the Miller-Madow correction.

**Intuition.** Entropy is concave and $\hat{p}$ is noisy, so Jensen pushes the estimate down.

**Solution.**

*Step 1.* The empirical distribution is unbiased, $\mathbb{E}[\hat{p}] = p$.

*Step 2.* $H$ is concave by Theorem 4.5, so Jensen's inequality applied to the random vector
$\hat{p}$ gives

$$
\mathbb{E}\!\left[H(\hat{p})\right] \le H\!\left(\mathbb{E}[\hat{p}]\right) = H(p),
$$

with strict inequality whenever $\hat{p}$ genuinely fluctuates.

*Step 3.* A second-order expansion of $H$ about $p$ gives the leading bias
$-\tfrac{K-1}{2n}$ nats, with $K$ the support size, which suggests the correction

$$
\hat{H}_{\mathrm{MM}} = \hat{H} + \frac{K-1}{2n} \text{ nats}.
$$

$$
\boxed{\mathbb{E}[\hat{H}] \le H(p), \qquad \hat{H}_{\mathrm{MM}} = \hat{H} + \tfrac{K-1}{2n} \text{ nats}}
$$

**Key takeaway.** Any entropy or mutual-information number computed from a contingency table must
report $n$ and $K$ alongside it, or it is not interpretable. Section 7.6 of the theory notebook
measures the bias against this prediction.

In [27]:
p_true = np.array([0.4, 0.25, 0.2, 0.1, 0.05])
K = p_true.size
H_true = H_nats(p_true)
print(f"true entropy = {H_true:.6f} nats\n")
for n in (20, 100, 500):
    est = np.array([H_nats(rng.multinomial(n, p_true) / n) for _ in range(4000)])
    print(f"n = {n:4d}:  E[H_hat] = {est.mean():.6f}   bias = {est.mean() - H_true:+.6f}"
          f"   predicted = {-(K - 1) / (2 * n):+.6f}"
          f"   Miller-Madow bias = {est.mean() + (K - 1) / (2 * n) - H_true:+.6f}")
    assert est.mean() < H_true

true entropy = 1.415023 nats

n =   20:  E[H_hat] = 1.302735   bias = -0.112288   predicted = -0.100000   Miller-Madow bias = -0.012288
n =  100:  E[H_hat] = 1.395333   bias = -0.019690   predicted = -0.020000   Miller-Madow bias = +0.000310
n =  500:  E[H_hat] = 1.410332   bias = -0.004690   predicted = -0.004000   Miller-Madow bias = -0.000690


### Problem L2.7 — Landauer's limit (physics)

**Statement.** Landauer's principle says erasing one bit of information at temperature $T$
dissipates at least $k_B T \ln 2$ joules. Compute that energy at $T = 300$ K, the energy to
erase one gibibyte, and compare with a stated CMOS switching energy of $10^{-16}$ J.

**Intuition.** Erasure destroys a distinction, and the second law charges for destroying
distinctions at the exchange rate $k_B T$ per nat.

**Solution.**

*Step 1.* One bit is $\ln 2$ nats, and the thermodynamic cost of one nat is $k_B T$, so the cost
of a bit is $k_B T \ln 2$.

*Step 2.* With $k_B = 1.380649 \times 10^{-23}$ J/K and $T = 300$ K,

$$
E_{\text{bit}} = 1.380649 \times 10^{-23} \times 300 \times 0.693147 = 2.8710 \times 10^{-21} \text{ J}.
$$

*Step 3.* One gibibyte is $2^{33} = 8.5899 \times 10^{9}$ bits, so erasing it costs

$$
2.8710 \times 10^{-21} \times 8.5899 \times 10^{9} = 2.4662 \times 10^{-11} \text{ J}.
$$

*Step 4.* Dividing $10^{-16}$ J by $E_{\text{bit}}$ gives $34\,831$: real logic dissipates about
four to five orders of magnitude more than the thermodynamic floor.

$$
\boxed{E_{\text{bit}} = 2.871 \times 10^{-21}\ \text{J}, \quad E_{\text{GiB}} = 2.466 \times 10^{-11}\ \text{J}}
$$

**Key takeaway.** Shannon entropy and thermodynamic entropy are the same quantity in different
units, and Landauer's constant $k_B T \ln 2$ is the exchange rate. Computation is not free, but
today's devices are nowhere near the limit that makes it expensive.

In [28]:
kB = 1.380649e-23
T = 300.0
E_bit = kB * T * np.log(2)
bits_per_gib = 8 * 2 ** 30
print(f"k_B T ln 2         = {E_bit:.6e} J per bit at {T:.0f} K")
print(f"bits in one gibibyte = {bits_per_gib:.6e}")
print(f"energy to erase 1 GiB = {E_bit * bits_per_gib:.6e} J")
print(f"CMOS 1e-16 J is       = {1e-16 / E_bit:.0f} times the Landauer floor")
print(f"erasing 1e18 bits/s costs {E_bit * 1e18 * 1e3:.4f} mW")
assert abs(E_bit - 2.871e-21) < 1e-24
assert abs(E_bit * bits_per_gib - 2.4662e-11) < 1e-14

k_B T ln 2         = 2.870979e-21 J per bit at 300 K
bits in one gibibyte = 8.589935e+09
energy to erase 1 GiB = 2.466152e-11 J
CMOS 1e-16 J is       = 34831 times the Landauer floor
erasing 1e18 bits/s costs 2.8710 mW


### Problem L2.8 — The Boltzmann distribution as maximum entropy (physics)

**Statement.** A three-level system has energies $E = (0, \varepsilon, 2\varepsilon)$ and known
mean energy $U = 0.9\varepsilon$. Find the maximum-entropy distribution over the levels and its
entropy in nats.

**Intuition.** Knowing only the mean energy, the least committal distribution is the one of
largest entropy subject to that mean — and problem L3.8 shows it is exponential in the energy.

**Solution.**

*Step 1.* By problem L3.8 the maximizer is $p_i = e^{-\beta E_i} / Z$ with
$Z = \sum_i e^{-\beta E_i}$, and $\beta$ fixed by the mean-energy constraint.

*Step 2.* In units of $\varepsilon$, write $x = e^{-\beta}$. The constraint
$\sum_i p_i E_i = 0.9$ becomes

$$
\frac{x + 2x^2}{1 + x + x^2} = 0.9 .
$$

*Step 3.* Rearranging gives $1.1x^2 + 0.1x - 0.9 = 0$, whose positive root is
$x = 0.860221$, so $\beta = -\ln x = 0.150566$ in units of $1/\varepsilon$.

*Step 4.* Then $Z = 1 + x + x^2 = 2.600201$ and

$$
p = (0.384586,\ 0.330829,\ 0.284586), \qquad H(p) = 1.0911 \text{ nats}.
$$

Compare the unconstrained maximum $\ln 3 = 1.0986$ nats: the mean-energy constraint costs only
$0.0075$ nats because $U = 0.9\varepsilon$ is close to the unconstrained mean $\varepsilon$.

$$
\boxed{p_i \propto e^{-\beta E_i} \text{ with } \beta = 0.1506/\varepsilon, \quad H = 1.0911 \text{ nats}}
$$

**Key takeaway.** The canonical ensemble is not a physical postulate but an inference: it is the
entropy maximizer at fixed mean energy, and $\beta = 1/k_B T$ is the Lagrange multiplier of that
constraint.

In [29]:
coef = [1.1, 0.1, -0.9]
roots = np.roots(coef)
x = float(roots[roots > 0].real.max())
beta = -np.log(x)
E = np.array([0.0, 1.0, 2.0])
w = np.exp(-beta * E)
p_boltz = w / w.sum()
print("positive root x   =", x, "  beta =", beta)
print("Z                 =", w.sum())
print("p                 =", p_boltz)
print("mean energy       =", float((p_boltz * E).sum()))
print("H(p)              =", H_nats(p_boltz), "nats    ln 3 =", np.log(3))

# every distribution with the same mean energy is p_boltz + t*(1, -2, 1), the only
# direction orthogonal to both the normalization and the energy constraint
d = np.array([1.0, -2.0, 1.0])
ts = np.linspace(-0.27, 0.15, 421)
cands = p_boltz[None, :] + ts[:, None] * d[None, :]
ents = np.array([H_nats(c) for c in cands])
print("\nsweeping the one-parameter family of feasible distributions")
print("  sum d =", d.sum(), "   sum E*d =", float((E * d).sum()), " (both zero, so the mean is preserved)")
print("  t at the entropy maximum =", ts[int(np.argmax(ents))])
print("  max entropy on the sweep =", ents.max(), "nats")
assert abs(float((p_boltz * E).sum()) - 0.9) < 1e-12
assert abs(H_nats(p_boltz) - 1.0911) < 5e-5
assert ents.max() <= H_nats(p_boltz) + 1e-12
assert abs(ts[int(np.argmax(ents))]) < 1e-9

positive root x   = 0.8602208565986943   beta = 0.15056611270614498
Z                 = 2.6002007787260855
p                 = [0.3846 0.3308 0.2846]
mean energy       = 0.9000000000000001
H(p)              = 1.0910981660684123 nats    ln 3 = 1.0986122886681098

sweeping the one-parameter family of feasible distributions
  sum d = 0.0    sum E*d = 0.0  (both zero, so the mean is preserved)
  t at the entropy maximum = 0.0
  max entropy on the sweep = 1.0910981660684123 nats


### Problem L2.9 — Gibbs entropy in physical units (physics)

**Statement.** A mole of independent two-state systems (say spin-$\tfrac12$ moments in a
magnetic field) has each system in state "up" with probability $p$. Give the molar entropy in
J/(K mol) at $p = \tfrac12$, the zero-field case, and at $p = \tfrac34$, a partially polarized
one.

**Intuition.** Gibbs entropy is Shannon entropy in nats multiplied by $k_B$; a mole multiplies
it by Avogadro's number, giving the gas constant $R$.

**Solution.**

*Step 1.* Gibbs' formula is $S = -k_B\sum_i p_i \ln p_i = k_B H_{\text{nats}}$.

*Step 2.* For $N_A$ independent systems the entropies add, so
$S_{\text{molar}} = N_A k_B H_b = R\,H_b$ with $R = 8.314463$ J/(K mol).

*Step 3.* At $p = \tfrac12$, $H_b = \ln 2 = 0.693147$ nats, so
$S = 8.314463 \times 0.693147 = 5.763146$ J/(K mol).

*Step 4.* At $p = \tfrac34$, $H_b = 0.562335$ nats, so $S = 4.675515$ J/(K mol).

$$
\boxed{S(\tfrac12) = R\ln 2 = 5.7631,\qquad S(\tfrac34) = 4.6755 \text{ J/(K mol)}}
$$

**Key takeaway.** $R \ln 2$ per mole is the textbook entropy of a two-level system, and it is
Theorem 4.4's maximum $\log K$ wearing thermodynamic units. Polarizing the spins reduces the
entropy exactly as it reduces the Shannon entropy.

In [30]:
R = 8.314462618
for p_up in (0.5, 0.75):
    Hb = H_nats([p_up, 1 - p_up])
    print(f"p = {p_up}:  H_b = {Hb:.6f} nats   S = {R * Hb:.6f} J/(K mol)")
print("R ln 2 =", R * np.log(2), "J/(K mol)")
assert abs(R * H_nats([0.5, 0.5]) - 5.763146) < 1e-5
assert abs(R * H_nats([0.75, 0.25]) - 4.675515) < 1e-5

p = 0.5:  H_b = 0.693147 nats   S = 5.763146 J/(K mol)
p = 0.75:  H_b = 0.562335 nats   S = 4.675515 J/(K mol)
R ln 2 = 5.763146321537762 J/(K mol)


### Problem L2.10 — A Huffman code against the entropy bound

**Statement.** For $p = (0.4, 0.2, 0.2, 0.1, 0.1)$ build the Huffman code, give
$\mathbb{E}[L]$, and compare it with $H(p)$, with the Shannon code, and with a fixed-length
code.

**Intuition.** Huffman merges the two rarest symbols repeatedly, which is exactly the greedy way
to spend the Kraft budget of Theorem 4.8.

**Solution.**

*Step 1.* $H(p) = 2.1219$ bits.

*Step 2.* Merge $0.1 + 0.1 = 0.2$; then merge the two $0.2$ leaves into $0.4$; then merge the
remaining $0.2$ with $0.4$ giving $0.6$; finally merge $0.4$ with $0.6$. The resulting depths are
$\ell = (2, 2, 2, 3, 3)$.

*Step 3.* The Kraft sum is $3(\tfrac14) + 2(\tfrac18) = 1$ exactly, so the tree is complete.

*Step 4.* $\mathbb{E}[L] = 0.4(2) + 0.2(2) + 0.2(2) + 0.1(3) + 0.1(3) = 2.2$ bits, against the
Shannon code's $2.8$ bits and a fixed-length code's $\lceil \log_2 5\rceil = 3$ bits.

$$
\boxed{\mathbb{E}[L_{\text{Huffman}}] = 2.2 \text{ bits}, \quad H(p) = 2.1219 \text{ bits}}
$$

**Key takeaway.** Huffman is optimal among symbol codes and lands $0.078$ bits above the entropy;
by Theorem 4.9 it can never be below it, and by Section 7.3 of the theory notebook the remaining
gap is removed only by coding blocks.

In [31]:
import heapq

p = np.array([0.4, 0.2, 0.2, 0.1, 0.1])


def huffman_lengths(p):
    heap = [(float(pi), i, [i]) for i, pi in enumerate(p)]
    heapq.heapify(heap)
    L = np.zeros(len(p), dtype=int)
    while len(heap) > 1:
        a = heapq.heappop(heap)
        b = heapq.heappop(heap)
        for i in a[2] + b[2]:
            L[i] += 1
        heapq.heappush(heap, (a[0] + b[0], min(a[1], b[1]), a[2] + b[2]))
    return L


Lh = huffman_lengths(p)
Ls = np.ceil(-np.log2(p)).astype(int)
print("H(p)              =", H_bits(p), "bits")
print("Huffman lengths   =", Lh, "  Kraft =", float((2.0 ** -Lh).sum()),
      "  E[L] =", float((p * Lh).sum()))
print("Shannon lengths   =", Ls, "  Kraft =", float((2.0 ** -Ls).sum()),
      "  E[L] =", float((p * Ls).sum()))
print("fixed length      =", int(np.ceil(np.log2(len(p)))), "bits")
assert abs(float((p * Lh).sum()) - 2.2) < 1e-12
assert H_bits(p) <= float((p * Lh).sum()) < H_bits(p) + 1
assert float((p * Lh).sum()) <= float((p * Ls).sum())

H(p)              = 2.1219280948873624 bits
Huffman lengths   = [2 2 2 3 3]   Kraft = 1.0   E[L] = 2.2
Shannon lengths   = [2 3 3 4 4]   Kraft = 0.625   E[L] = 2.8
fixed length      = 3 bits


### Problem L2.11 — Bits per token as a corpus size

**Statement.** A model scores $1.8$ bits per token on a corpus of $2 \times 10^{9}$ tokens.
How large is the corpus once compressed with an ideal code built from that model, and how does
that compare with storing the tokens as $4$-byte integers?

**Intuition.** Theorem 4.9 makes an average code length a file size: multiply bits per symbol by
the number of symbols.

**Solution.**

*Step 1.* Total ideal code length is
$1.8 \times 2 \times 10^{9} = 3.6 \times 10^{9}$ bits.

*Step 2.* Dividing by $8$ gives $4.5 \times 10^{8}$ bytes, that is $450$ megabytes.

*Step 3.* Storing raw $4$-byte token ids costs $8 \times 10^{9}$ bytes, or $8$ gigabytes.

*Step 4.* The ratio is $8 / 0.45 = 17.8$.

$$
\boxed{450 \text{ MB compressed against } 8 \text{ GB raw, a factor of } 17.8}
$$

**Key takeaway.** A better model is literally a better compressor, and the improvement is
measured in the same units. Reducing the score by $0.1$ bits per token saves $25$ MB on this
corpus.

In [32]:
bits_per_token = 1.8
tokens = 2e9
total_bits = bits_per_token * tokens
print(f"compressed size = {total_bits / 8 / 1e6:.1f} MB")
print(f"raw 4-byte ids  = {tokens * 4 / 1e9:.1f} GB")
print(f"ratio           = {tokens * 4 / (total_bits / 8):.1f}x")
print(f"saving from 0.1 bits/token = {0.1 * tokens / 8 / 1e6:.1f} MB")
assert abs(total_bits / 8 / 1e6 - 450.0) < 1e-9
assert abs(tokens * 4 / (total_bits / 8) - 17.7778) < 1e-3

compressed size = 450.0 MB
raw 4-byte ids  = 8.0 GB
ratio           = 17.8x
saving from 0.1 bits/token = 25.0 MB


### Problem L2.12 — Predictive entropy as an uncertainty score

**Statement.** Three inputs produce logits $(3,0,0)$, $(2,1.9,0.1)$ and $(1,0.9,0.8)$. Rank them
by predictive entropy in bits and say what the ranking is used for.

**Intuition.** A confident model concentrates its softmax mass, and Theorem 4.4 says
concentration means low entropy.

**Solution.**

*Step 1.* Softmax of $(3,0,0)$ is $(0.9094, 0.0453, 0.0453)$, with $H = 0.5289$ bits.

*Step 2.* Softmax of $(2, 1.9, 0.1)$ is $(0.4868, 0.4404, 0.0728)$, with $H = 1.3018$ bits.

*Step 3.* Softmax of $(1, 0.9, 0.8)$ is $(0.3672, 0.3322, 0.3006)$, with $H = 1.5802$ bits,
close to the ceiling $\log_2 3 = 1.5850$ bits.

*Step 4.* The ranking from most to least uncertain is the third, then the second, then the
first.

$$
\boxed{H = 1.5802 \gt 1.3018 \gt 0.5289 \text{ bits}}
$$

**Key takeaway.** Predictive entropy is the standard acquisition score in active learning and a
common out-of-distribution signal. It measures the spread of the model's own belief, so it
detects ambiguity but not miscalibration: a confidently wrong model scores low.

In [33]:
cases = {"(3, 0, 0)": [3.0, 0.0, 0.0],
         "(2, 1.9, 0.1)": [2.0, 1.9, 0.1],
         "(1, 0.9, 0.8)": [1.0, 0.9, 0.8]}
scores = {}
for name, z in cases.items():
    z = np.array(z)
    p = np.exp(z - logsumexp(z))
    scores[name] = H_bits(p)
    print(f"logits {name:<15} p = {np.round(p, 4)}   H = {H_bits(p):.4f} bits")
print("ceiling log2 3 =", np.log2(3), "bits")
print("ranking (most uncertain first):", sorted(scores, key=scores.get, reverse=True))
assert scores["(1, 0.9, 0.8)"] > scores["(2, 1.9, 0.1)"] > scores["(3, 0, 0)"]
assert max(scores.values()) < np.log2(3)

logits (3, 0, 0)       p = [0.9094 0.0453 0.0453]   H = 0.5289 bits
logits (2, 1.9, 0.1)   p = [0.4868 0.4404 0.0728]   H = 1.3018 bits
logits (1, 0.9, 0.8)   p = [0.3672 0.3322 0.3006]   H = 1.5802 bits
ceiling log2 3 = 1.584962500721156 bits
ranking (most uncertain first): ['(1, 0.9, 0.8)', '(2, 1.9, 0.1)', '(3, 0, 0)']


## L3 — Challenge Proofs

### Problem L3.1 — Uniqueness of entropy from the grouping axiom

**Statement.** Let $f(K)$ be the uncertainty of a uniform distribution on $K$ outcomes. Assume
$f(Km) = f(K) + f(m)$ for all integers $K, m \ge 1$ and that $f$ is non-decreasing with
$f(2) \gt f(1)$. Prove $f(K) = c\log K$ with $c \gt 0$, then extend to all rational
distributions.

**Intuition.** Additivity on integers plus order is enough to force a logarithm, because a
logarithm is the only monotone solution of that functional equation.

**Solution.**

*Step 1.* Setting $m = 1$ gives $f(1) = 0$, and induction on additivity gives
$f(K^{n}) = n f(K)$. Monotonicity with $f(2) \gt f(1) = 0$ makes $f(m) \gt 0$ for $m \ge 2$.

*Step 2.* Fix $K, m \ge 2$ and $n \ge 1$, and pick the integer $r \ge 0$ with

$$
m^{r} \le K^{n} \lt m^{r+1}.
$$

Taking logarithms and dividing by $n \log m$,

$$
\frac{r}{n} \le \frac{\log K}{\log m} \lt \frac{r+1}{n}.
$$

*Step 3.* Applying the non-decreasing $f$ to the same sandwich and using Step 1,

$$
r f(m) \le n f(K) \le (r+1) f(m),
\qquad\text{so}\qquad
\frac{r}{n} \le \frac{f(K)}{f(m)} \le \frac{r+1}{n}.
$$

*Step 4.* Both $f(K)/f(m)$ and $\log K/\log m$ sit in an interval of width $1/n$ for every $n$,
so they are equal, giving $f(K) = c \log K$ with $c = f(m)/\log m \gt 0$.

*Step 5 (the rational extension, done explicitly).* Take a rational distribution
$p_k = n_k/n$ with $n = n_1 + \dots + n_K$. Apply the grouping axiom to a *uniform* experiment on
$n$ outcomes partitioned into blocks of sizes $n_1, \dots, n_K$: stage one picks the block, with
probabilities $p_k$; stage two picks uniformly inside it. Hence

$$
f(n) = H(p_1, \dots, p_K) + \sum_{k=1}^{K} p_k f(n_k).
$$

*Step 6.* Substituting $f = c\log$ and solving for $H$,

$$
H(p) = c \log n - c\sum_k p_k \log n_k = -c \sum_k p_k \log \frac{n_k}{n} = -c\sum_k p_k \log p_k .
$$

Continuity then carries the formula from the rationals, which are dense in the simplex, to every
distribution.

$$
\boxed{f(K) = c\log K \ \Longrightarrow\ H(p) = -c\sum_k p_k \log p_k}
$$

**Key takeaway.** This is Steps 3 to 5 of Proof 5.3 in isolation. The axioms are not vacuous: the
Gini index $G(p) = 1 - \sum_k p_k^2$ is symmetric, continuous and maximized by the uniform
distribution, yet the code cell shows it violates grouping — which is why Gini and entropy give
different trees.

In [34]:
def gini(p):
    p = np.asarray(p, dtype=float)
    return float(1.0 - (p ** 2).sum())


p = np.array([0.5, 0.3, 0.2])
w = p[1] + p[2]
sub = np.array([p[1] / w, p[2] / w])
print("entropy grouping check")
print("  H(p)                      =", H_bits(p))
print("  H(0.5, 0.5) + 0.5 H(sub)  =", H_bits([p[0], w]) + w * H_bits(sub))
print(f"  residual                  = {abs(H_bits(p) - (H_bits([p[0], w]) + w * H_bits(sub))):.3e}")
print("\nGini grouping check")
print("  G(p)                      =", gini(p))
print("  G(0.5, 0.5) + 0.5 G(sub)  =", gini([p[0], w]) + w * gini(sub))
print("  defect                    =", gini(p) - (gini([p[0], w]) + w * gini(sub)))
assert abs(H_bits(p) - (H_bits([p[0], w]) + w * H_bits(sub))) < 1e-12
assert abs(gini(p) - (gini([p[0], w]) + w * gini(sub))) > 0.01

f = lambda K: np.log2(K)
for K, m in [(3, 5), (7, 4), (6, 6)]:
    assert abs(f(K * m) - (f(K) + f(m))) < 1e-12
print("\nf(K) = log2 K satisfies f(Km) = f(K) + f(m) for every pair tested")

entropy grouping check
  H(p)                      = 1.4854752972273344
  H(0.5, 0.5) + 0.5 H(sub)  = 1.4854752972273344
  residual                  = 0.000e+00

Gini grouping check
  G(p)                      = 0.62
  G(0.5, 0.5) + 0.5 G(sub)  = 0.74
  defect                    = -0.12

f(K) = log2 K satisfies f(Km) = f(K) + f(m) for every pair tested


### Problem L3.2 — Fano's inequality

**Statement.** Let $X$ take $K$ values, let $\hat{X}$ be any estimate of $X$ on the same
alphabet, and let $P_e = \mathbb{P}(\hat{X} \neq X)$. Prove

$$
H(X \mid \hat{X}) \le H_b(P_e) + P_e \log(K-1).
$$

**Intuition.** If the estimate is usually right, the residual uncertainty is only "which of the
rare error cases occurred", and there are at most $K-1$ of them.

**Solution.**

*Step 0 (the two facts used, stated here so the argument is self-contained).* For jointly
distributed $A, B$ the conditional entropy is
$H(A \mid B) = -\sum_{a,b} p(a,b)\log p(a \mid b)$, and it satisfies

- the **chain rule** $H(A, B \mid C) = H(B \mid C) + H(A \mid B, C)$, which follows from
  $p(a,b \mid c) = p(b \mid c)\,p(a \mid b, c)$ and splitting the logarithm;
- **conditioning reduces entropy**, $H(A \mid B) \le H(A)$, which is Lemma 4.1 applied to
  $p(a,b)$ against $p(a)p(b)$.

[Module 02](../02_joint_and_conditional_entropy/) develops both systematically; nothing below
needs more than the two lines above.

*Step 1.* Let $E = \mathbf{1}\{\hat{X} \neq X\}$ and expand $H(E, X \mid \hat{X})$ in two orders:

$$
H(E, X \mid \hat{X}) = H(E \mid \hat{X}) + H(X \mid E, \hat{X}) = H(X \mid \hat{X}) + H(E \mid X, \hat{X}).
$$

*Step 2.* $E$ is a deterministic function of $(X, \hat{X})$, so $H(E \mid X, \hat{X}) = 0$ and

$$
H(X \mid \hat{X}) = H(E \mid \hat{X}) + H(X \mid E, \hat{X}).
$$

*Step 3.* Conditioning reduces entropy, so $H(E \mid \hat{X}) \le H(E) = H_b(P_e)$.

*Step 4.* Split the second term on the value of $E$. Given $E = 0$ we have $X = \hat{X}$, so that
term is $0$. Given $E = 1$, $X$ lies in the $K-1$ values other than $\hat{X}$, so by Theorem 4.4
its conditional entropy is at most $\log(K-1)$. Averaging,

$$
H(X \mid E, \hat{X}) \le P_e \log(K-1).
$$

*Step 5.* Adding Steps 3 and 4 gives the inequality. Rearranged with
$H_b(P_e) \le \log 2$, it becomes a lower bound on the error probability:

$$
P_e \ge \frac{H(X \mid \hat{X}) - \log 2}{\log(K-1)} .
$$

$$
\boxed{H(X \mid \hat{X}) \le H_b(P_e) + P_e\log(K-1)}
$$

**Key takeaway.** Accuracy is capped by information. A classifier whose features leave
$H(X \mid \hat{X})$ bits of residual uncertainty cannot be accurate, whatever its architecture,
and this is the converse workhorse behind minimax lower bounds in statistics.

In [35]:
K = 5
p_x = rng.dirichlet(np.ones(K))
worst = np.inf
for trial in range(2000):
    channel = rng.dirichlet(np.ones(K), size=K)
    joint = p_x[:, None] * channel
    p_hat = joint.sum(axis=0)
    cond = -(joint * np.log2(np.where(joint > 0, joint / p_hat[None, :], 1.0))).sum()
    Pe = float(joint.sum() - np.trace(joint))
    rhs = H_bits([Pe, 1 - Pe]) + Pe * np.log2(K - 1)
    worst = min(worst, rhs - cond)
print(f"K = {K}, 2000 random channels")
print(f"smallest slack  H_b(Pe) + Pe log2(K-1) - H(X | Xhat) = {worst:.6f} bits")
assert worst >= -1e-12

K = 5, 2000 random channels
smallest slack  H_b(Pe) + Pe log2(K-1) - H(X | Xhat) = 0.404594 bits


### Problem L3.3 — Maximum entropy under a mean constraint

**Statement.** Among distributions on $\{0, 1, 2, \dots\}$ with fixed mean $\mu \gt 0$, find the
entropy maximizer and its entropy in nats.

**Intuition.** One linear constraint buys one exponential factor, so the answer is geometric.

**Solution.**

*Step 1.* By problem L3.8 with $g(k) = k$, the maximizer is $p_k \propto e^{-\beta k}$, that is
$p_k = (1-q)q^{k}$ with $q = e^{-\beta} \in (0,1)$.

*Step 2.* Matching the mean: problem L1.3 gives $\mathbb{E}[X] = q/(1-q)$, so $q/(1-q) = \mu$
and $q = \mu/(1+\mu)$.

*Step 3.* Substituting into the entropy formula of problem L1.3,
$H = H_b(q)/(1-q)$ with $H_b$ in nats. With $1 - q = 1/(1+\mu)$ and $q = \mu/(1+\mu)$,

$$
H = (1+\mu)\left[-\frac{1}{1+\mu}\ln\frac{1}{1+\mu} - \frac{\mu}{1+\mu}\ln\frac{\mu}{1+\mu}\right]
= (1+\mu)\ln(1+\mu) - \mu\ln\mu .
$$

*Step 4.* Normalizability forces $q \lt 1$, which holds for every finite $\mu \gt 0$.

$$
\boxed{p_k^{\star} = \frac{1}{1+\mu}\left(\frac{\mu}{1+\mu}\right)^{k}, \qquad H = (1+\mu)\ln(1+\mu) - \mu\ln\mu \text{ nats}}
$$

**Key takeaway.** Every constraint type has its own maximum-entropy family: support size gives
uniform (Theorem 4.4), a mean on the non-negative integers gives geometric, a variance on
$\mathbb{R}$ gives Gaussian (Theorem 4.7), a mean energy gives Boltzmann (problem L2.8).

In [36]:
for mu in (0.5, 2.0, 7.0):
    q = mu / (1 + mu)
    k = np.arange(0, 400000)
    pk = (1 - q) * q ** k
    closed = (1 + mu) * np.log(1 + mu) - mu * np.log(mu)
    print(f"mu = {mu}:  q = {q:.6f}   mean = {float((k * pk).sum()):.6f}"
          f"   H direct = {H_nats(pk):.9f}   closed = {closed:.9f} nats")
    assert abs(float((k * pk).sum()) - mu) < 1e-6
    assert abs(H_nats(pk) - closed) < 1e-6

# on the finite support {0, ..., 12} the maximizer is the truncated geometric of L3.8
support = np.arange(13, dtype=float)
lo_b, hi_b = -5.0, 5.0
for _ in range(200):
    mid = 0.5 * (lo_b + hi_b)
    wm = np.exp(-mid * support)
    if float((wm / wm.sum() * support).sum()) > 2.0:
        lo_b = mid
    else:
        hi_b = mid
beta = 0.5 * (lo_b + hi_b)
q_star = np.exp(-beta * support)
q_star /= q_star.sum()
print(f"\ntruncated support 0..12 with mean 2: exp(-beta) = {np.exp(-beta):.6f}"
      f"   (untruncated q = 2/3 = 0.666667)")
print(f"  mean      = {float((q_star * support).sum()):.9f}")
print(f"  H(q_star) = {H_nats(q_star):.9f} nats")

u = support - support.mean()
best_pert = -np.inf
for _ in range(20000):
    v = rng.standard_normal(13)
    v -= v.mean()
    v -= (v @ u) / (u @ u) * u
    if np.all(v >= 0):
        continue
    t = 0.5 * float(np.min(q_star[v < 0] / (-v[v < 0])))
    cand = q_star + t * v
    best_pert = max(best_pert, H_nats(cand))
print(f"  best mean-preserving perturbation: H = {best_pert:.9f} nats"
      f"   (deficit {H_nats(q_star) - best_pert:.3e})")
assert abs(float((q_star * support).sum()) - 2.0) < 1e-9
assert best_pert < H_nats(q_star)

mu = 0.5:  q = 0.333333   mean = 0.500000   H direct = 0.954771252   closed = 0.954771252 nats


mu = 2.0:  q = 0.666667   mean = 2.000000   H direct = 1.909542505   closed = 1.909542505 nats

mu = 7.0:  q = 0.875000   mean = 7.000000   H direct = 3.014161290   closed = 3.014161290 nats

truncated support 0..12 with mean 2: exp(-beta) = 0.675259   (untruncated q = 2/3 = 0.666667)
  mean      = 2.000000000
  H(q_star) = 1.903957793 nats


  best mean-preserving perturbation: H = 1.903471514 nats   (deficit 4.863e-04)

### Problem L3.4 — Asymptotic equipartition property and the typical set

**Statement.** Let $X_1, \dots, X_n$ be i.i.d. with entropy $H$ in nats on a finite alphabet.
Prove that $-\tfrac1n \ln p(X_1, \dots, X_n) \to H$ in probability and deduce that about
$e^{nH}$ sequences carry almost all of the probability.

**Intuition.** The log-probability of a long sequence is a sum of i.i.d. terms, so the law of
large numbers applies to it.

**Solution.**

*Step 1.* Independence factorizes the joint pmf, so

$$
-\frac{1}{n}\ln p(X_1, \dots, X_n) = \frac{1}{n}\sum_{i=1}^{n} Z_i, \qquad Z_i = -\ln p(X_i),
$$

with $\mathbb{E}[Z_1] = H$ by Definition 3.2.

*Step 2.* On a finite alphabet $Z_i \le \ln(1/p_{\min}) \lt \infty$, so
$\sigma^2 = \operatorname{Var}(Z_1)$ is finite and Chebyshev gives

$$
\mathbb{P}\left(\left\lvert \tfrac1n\textstyle\sum_i Z_i - H \right\rvert \gt \epsilon\right) \le \frac{\sigma^2}{n\epsilon^2} \to 0 .
$$

That is the convergence claim, and it says $\mathbb{P}(A_{\epsilon}^{(n)}) \to 1$ for the typical
set of Definition 3.7.

*Step 3 (upper bound on the count).* Every typical sequence has probability at least
$e^{-n(H+\epsilon)}$ and the total probability is at most $1$, so

$$
\left\lvert A_{\epsilon}^{(n)}\right\rvert \le e^{n(H+\epsilon)} .
$$

*Step 4 (lower bound).* For $n$ large enough $\mathbb{P}(A_{\epsilon}^{(n)}) \ge 1-\delta$, and
each typical sequence has probability at most $e^{-n(H-\epsilon)}$, so

$$
\left\lvert A_{\epsilon}^{(n)}\right\rvert \ge (1-\delta)\,e^{n(H-\epsilon)} .
$$

*Step 5.* Indexing only the typical set therefore costs $n(H+\epsilon)$ nats and fails with
probability tending to zero: this is the achievability half of the source coding theorem,
obtained without constructing a code.

$$
\boxed{(1-\delta)e^{n(H-\epsilon)} \le \left\lvert A_{\epsilon}^{(n)}\right\rvert \le e^{n(H+\epsilon)}}
$$

**Key takeaway.** Entropy is the exponential growth rate of the set of sequences that actually
occur. Section 7.2 of the theory notebook measures the $n^{-1/2}$ rate at which the
concentration tightens.

In [37]:
from math import comb

p1 = 0.3
H_ber = H_nats([p1, 1 - p1])
print(f"H = {H_ber:.6f} nats\n")
print("  n     |A_eps|        P(A_eps)     lower bound     upper bound")
for n in (25, 50, 100):
    eps = 0.1
    ks = [k for k in range(n + 1)
          if abs(-(k * np.log(p1) + (n - k) * np.log(1 - p1)) / n - H_ber) <= eps]
    size = sum(comb(n, k) for k in ks)
    prob = sum(comb(n, k) * p1 ** k * (1 - p1) ** (n - k) for k in ks)
    lo, hi = np.exp(n * (H_ber - eps)), np.exp(n * (H_ber + eps))
    print(f"{n:4d}   {size:.4e}    {prob:.6f}     {lo:.4e}      {hi:.4e}")
    assert lo <= size <= hi
    assert prob > 0.8

H = 0.610864 nats

  n     |A_eps|        P(A_eps)     lower bound     upper bound
  25   7.1042e+06    0.811728     3.5208e+05      5.2253e+07
  50   1.1407e+14    0.912005     1.2396e+11      2.7304e+15
 100   5.6173e+28    0.988304     1.5366e+22      7.4549e+30


### Problem L3.5 — Entropy power and the entropy power inequality

**Statement.** The **entropy power** of a real random variable is

$$
N(X) = \frac{e^{2h(X)}}{2\pi e} \quad \text{(with } h \text{ in nats)}.
$$

Show $N(X) \le \operatorname{Var}(X)$ with equality only for Gaussians, and verify the entropy
power inequality $N(X+Y) \ge N(X) + N(Y)$ for independent $X, Y$, first for two Gaussians and
then for two uniforms.

**Intuition.** $N(X)$ is the variance a Gaussian would need to have the same differential
entropy — a "variance measured in entropy units".

**Solution.**

*Step 1.* Theorem 4.7 gives $h(X) \le \tfrac12\ln(2\pi e \sigma^2)$. Exponentiating and dividing
by $2\pi e$,

$$
N(X) = \frac{e^{2h(X)}}{2\pi e} \le \frac{2\pi e \sigma^2}{2\pi e} = \operatorname{Var}(X),
$$

with equality exactly in the equality case of Theorem 4.7, that is for Gaussians.

*Step 2 (Gaussian case of the EPI).* If $X \sim \mathcal{N}(0, \sigma_1^2)$ and
$Y \sim \mathcal{N}(0, \sigma_2^2)$ are independent then $N(X) = \sigma_1^2$,
$N(Y) = \sigma_2^2$ and $X+Y \sim \mathcal{N}(0, \sigma_1^2+\sigma_2^2)$, so

$$
N(X+Y) = \sigma_1^2 + \sigma_2^2 = N(X) + N(Y) .
$$

Gaussians attain the inequality with equality.

*Step 3 (uniform case).* Let $X, Y$ be independent uniform on $(-a, a)$ with $a = \sqrt{3}$, so
each has variance $1$ and $h = \ln 2a = 1.2425$ nats, giving $N(X) = N(Y) = 0.7026$.

*Step 4.* The sum is triangular on $(-2a, 2a)$ with density $(2a - \lvert x \rvert)/(4a^2)$, and
its differential entropy is $h(X+Y) = \ln 2a + \tfrac12 = 1.7425$ nats, so
$N(X+Y) = 1.9099$.

*Step 5.* Comparing, $1.9099 \ge 0.7026 + 0.7026 = 1.4052$: the inequality is strict, as expected
for non-Gaussian summands.

$$
\boxed{N(X) \le \operatorname{Var}(X), \qquad N(X+Y) \ge N(X) + N(Y)}
$$

**Key takeaway.** The entropy power inequality is the entropy analogue of "variances add", and
its failure to be an equality measures how far a variable is from Gaussian. It is the tool behind
capacity converses for non-Gaussian noise.

In [38]:
def entropy_power(h):
    return float(np.exp(2 * h) / (2 * np.pi * np.e))


s1, s2 = 1.0, np.sqrt(3.0)
hg = lambda s: 0.5 * np.log(2 * np.pi * np.e * s * s)
print("two Gaussians, variances 1 and 3")
print("  N(X), N(Y)  =", entropy_power(hg(s1)), entropy_power(hg(s2)))
print("  N(X+Y)      =", entropy_power(hg(np.sqrt(s1 ** 2 + s2 ** 2))))
assert abs(entropy_power(hg(np.sqrt(s1 ** 2 + s2 ** 2)))
           - (entropy_power(hg(s1)) + entropy_power(hg(s2)))) < 1e-9

a = np.sqrt(3.0)
h_unif = np.log(2 * a)
h_tri = np.log(2 * a) + 0.5
num, _ = quad(lambda x: -((2 * a - abs(x)) / (4 * a * a))
              * np.log((2 * a - abs(x)) / (4 * a * a)),
              -2 * a + 1e-12, 2 * a - 1e-12, points=[0.0], limit=400)
print("\ntwo uniforms on (-sqrt3, sqrt3), each of variance 1")
print("  h(uniform)          =", h_unif, "nats   N =", entropy_power(h_unif))
print("  h(triangular) quad  =", num, "   closed form =", h_tri)
print("  N(X+Y)              =", entropy_power(h_tri))
print("  N(X) + N(Y)         =", 2 * entropy_power(h_unif))
print("  Var(X+Y)            =", 2.0, " >= N(X+Y):", 2.0 >= entropy_power(h_tri))
assert abs(num - h_tri) < 1e-8
assert entropy_power(h_tri) > 2 * entropy_power(h_unif)
assert entropy_power(h_unif) < 1.0

two Gaussians, variances 1 and 3
  N(X), N(Y)  = 1.0 2.9999999999999996
  N(X+Y)      = 3.999999999999997

two uniforms on (-sqrt3, sqrt3), each of variance 1
  h(uniform)          = 1.2424533248940002 nats   N = 0.70259797829183
  h(triangular) quad  = 1.7424533248940006    closed form = 1.7424533248940002
  N(X+Y)              = 1.9098593171027443
  N(X) + N(Y)         = 1.40519595658366
  Var(X+Y)            = 2.0  >= N(X+Y): True


### Problem L3.6 — Type counting: $\binom{n}{np}$ against $2^{nH_b(p)}$

**Statement.** For $np$ an integer, prove

$$
\frac{2^{n H_b(p)}}{n+1} \;\le\; \binom{n}{np} \;\le\; 2^{n H_b(p)},
$$

and deduce $\tfrac1n \log_2 \binom{n}{np} \to H_b(p)$.

**Intuition.** The binomial coefficient counts the sequences of a given composition, and by the
AEP that count must grow like $2^{nH}$.

**Solution.**

*Step 1 (upper bound).* The binomial probability of $k = np$ successes is at most one:

$$
1 \ge \binom{n}{np} p^{np}(1-p)^{n(1-p)} .
$$

*Step 2.* Taking $\log_2$ of the product term,

$$
\log_2\left[p^{np}(1-p)^{n(1-p)}\right] = n\left[p\log_2 p + (1-p)\log_2(1-p)\right] = -n H_b(p),
$$

so $\binom{n}{np} \le 2^{n H_b(p)}$.

*Step 3 (lower bound).* Among the $n+1$ binomial terms
$b_k = \binom{n}{k}p^{k}(1-p)^{n-k}$, the largest is at $k = np$, because
$b_{k+1}/b_k = \tfrac{(n-k)p}{(k+1)(1-p)}$ crosses $1$ exactly there.

*Step 4.* The $n+1$ terms sum to $1$, so the largest is at least $1/(n+1)$:

$$
\binom{n}{np}p^{np}(1-p)^{n(1-p)} \ge \frac{1}{n+1},
$$

which rearranges to $\binom{n}{np} \ge 2^{nH_b(p)}/(n+1)$.

*Step 5.* Taking $\tfrac1n\log_2$ of both bounds gives
$H_b(p) - \tfrac{\log_2(n+1)}{n} \le \tfrac1n\log_2\binom{n}{np} \le H_b(p)$, and the
correction vanishes.

$$
\boxed{\frac{1}{n}\log_2 \binom{n}{np} \to H_b(p)}
$$

**Key takeaway.** Entropy is a counting exponent. This is the combinatorial face of
Theorem 4.10: the typical set for a Bernoulli source is essentially the set of sequences with
the right number of ones, and there are $2^{nH_b(p)}$ of them up to a polynomial factor.

In [39]:
from math import comb, lgamma

p = 0.3
Hb = H_bits([p, 1 - p])
print(f"H_b(0.3) = {Hb:.6f} bits\n")
print("   n      (1/n) log2 C(n, 0.3n)     lower bound        upper bound")
for n in (10, 100, 1000, 10000):
    k = int(round(n * p))
    log2C = (lgamma(n + 1) - lgamma(k + 1) - lgamma(n - k + 1)) / np.log(2)
    lo = Hb - np.log2(n + 1) / n
    print(f"{n:6d}        {log2C / n:.8f}          {lo:.8f}         {Hb:.8f}")
    assert lo - 1e-9 <= log2C / n <= Hb + 1e-9
assert abs(comb(100, 30) - np.exp(lgamma(101) - lgamma(31) - lgamma(71))) / comb(100, 30) < 1e-9

H_b(0.3) = 0.881291 bits

   n      (1/n) log2 C(n, 0.3n)     lower bound        upper bound
    10        0.69068906          0.53534774         0.88129090
   100        0.84602661          0.81470878         0.88129090
  1000        0.87610758          0.87132367         0.88129090
 10000        0.88060651          0.87996211         0.88129090


### Problem L3.7 — McMillan's inequality: unique decodability already forces Kraft

**Statement.** Let a $D$-ary code have lengths $\ell_1, \dots, \ell_K$ and be **uniquely
decodable**, meaning distinct source strings map to distinct concatenations. Prove
$\sum_k D^{-\ell_k} \le 1$.

**Intuition.** Raise the Kraft sum to the $n$-th power: it becomes a count of source
$n$-strings by total codeword length, and unique decodability caps that count by the number of
available strings.

**Solution.**

*Step 1.* Put $S = \sum_{k} D^{-\ell_k}$ and expand the $n$-th power:

$$
S^{n} = \sum_{k_1, \dots, k_n} D^{-(\ell_{k_1} + \dots + \ell_{k_n})} = \sum_{m} N(m)\, D^{-m},
$$

where $N(m)$ counts source $n$-strings whose concatenated codeword has total length $m$.

*Step 2.* Unique decodability makes the map from source $n$-strings to $D$-ary strings
injective, so $N(m) \le D^{m}$, the number of $D$-ary strings of length $m$.

*Step 3.* The total length lies between $n\ell_{\min}$ and $n\ell_{\max}$, so

$$
S^{n} \le \sum_{m = n\ell_{\min}}^{n\ell_{\max}} D^{m}D^{-m} \le n\,\ell_{\max}.
$$

*Step 4.* Hence $S \le (n \ell_{\max})^{1/n}$ for every $n$, and the right-hand side tends to
$1$ as $n \to \infty$. Therefore $S \le 1$.

$$
\boxed{\text{Uniquely decodable} \implies \sum_k D^{-\ell_k} \le 1}
$$

**Key takeaway.** Dropping prefix-freeness for the weaker unique decodability buys nothing: the
achievable length profiles are the same set. So the coding converse of Theorem 4.9 applies to
every uniquely decodable code, and there is no reason ever to use a non-prefix code.

In [40]:
import itertools

words = ["0", "01", "011"]
kraft = float(sum(2.0 ** -len(w) for w in words))
print("code            :", words)
print("prefix-free     :", all(not b.startswith(a) for a in words for b in words if a is not b))
print("Kraft sum       :", kraft, " <= 1 as McMillan predicts")

seen = {}
collision = None
for L in range(1, 7):
    for combo in itertools.product(range(3), repeat=L):
        s = "".join(words[i] for i in combo)
        if s in seen and seen[s] != combo:
            collision = (s, seen[s], combo)
        seen[s] = combo
print("distinct concatenations up to length 6 :", len(seen))
print("decoding collision found               :", collision)
assert kraft <= 1.0
assert collision is None

bad = ["0", "1", "01"]
print("\ncode", bad, "has Kraft sum", float(sum(2.0 ** -len(w) for w in bad)), "> 1")
print("  '01' is ambiguous:", "".join([bad[0], bad[1]]), "==", bad[2])
assert float(sum(2.0 ** -len(w) for w in bad)) > 1.0

code            : ['0', '01', '011']
prefix-free     : False
Kraft sum       : 0.875  <= 1 as McMillan predicts
distinct concatenations up to length 6 : 1092
decoding collision found               : None

code ['0', '1', '01'] has Kraft sum 1.25 > 1
  '01' is ambiguous: 01 == 01


### Problem L3.8 — Maximum entropy under a linear constraint is an exponential family

**Statement.** Fix a function $g$ on a finite alphabet and a value $\mu$ in the interior of the
range of $\mathbb{E}[g]$. Among all $p$ with $\mathbb{E}_p[g] = \mu$, show the entropy maximizer
is

$$
q_k = \frac{e^{-\beta g(k)}}{Z(\beta)}, \qquad Z(\beta) = \sum_j e^{-\beta g(j)},
$$

with $\beta$ chosen so that $\mathbb{E}_q[g] = \mu$.

**Intuition.** Gibbs' inequality compares any feasible $p$ against the exponential candidate, and
the constraint makes the cross term identical for both.

**Solution.**

*Step 1.* Choose $\beta$ so that $\mathbb{E}_q[g] = \mu$. This is possible because
$\beta \mapsto \mathbb{E}_q[g]$ is continuous and strictly decreasing, with
$-\tfrac{d}{d\beta}\mathbb{E}_q[g] = \operatorname{Var}_q(g) \gt 0$, and it sweeps the interior of
the range of $g$.

*Step 2.* For any feasible $p$, expand the divergence of $p$ from $q$ in nats:

$$
D_{\mathrm{KL}}(p \parallel q) = -H(p) - \sum_k p_k \ln q_k .
$$

*Step 3.* Substitute $\ln q_k = -\beta g(k) - \ln Z$:

$$
-\sum_k p_k \ln q_k = \beta \sum_k p_k g(k) + \ln Z = \beta\mu + \ln Z,
$$

which depends on $p$ only through the constraint, hence takes the same value for $p$ and for $q$.

*Step 4.* Therefore $H(p) = \beta\mu + \ln Z - D_{\mathrm{KL}}(p \parallel q)$, and by Lemma 4.1
the divergence is non-negative:

$$
H(p) \le \beta\mu + \ln Z = H(q),
$$

with equality if and only if $p = q$.

$$
\boxed{q_k \propto e^{-\beta g(k)}, \qquad H(q) = \beta\mu + \ln Z(\beta) \text{ nats}}
$$

**Key takeaway.** Every maximum-entropy problem with linear constraints has an exponential-family
answer, and $\beta$ is the Lagrange multiplier of the constraint. Uniform, geometric, Boltzmann
and Gaussian are all the same computation with different $g$.

In [41]:
g = np.array([0.0, 1.0, 2.0, 3.5])
mu = 1.4


def mean_at(beta):
    w = np.exp(-beta * g)
    return float((w / w.sum() * g).sum())


lo, hi = -20.0, 20.0
for _ in range(200):
    mid = 0.5 * (lo + hi)
    if mean_at(mid) > mu:
        lo = mid
    else:
        hi = mid
beta = 0.5 * (lo + hi)
w = np.exp(-beta * g)
q = w / w.sum()
print("beta      =", beta)
print("q         =", q)
print("E_q[g]    =", mean_at(beta), " target", mu)
print("H(q)      =", H_nats(q), "nats")
print("beta*mu + ln Z =", beta * mu + np.log(w.sum()), "nats")

# exactly feasible competitors: perturb q in directions orthogonal to 1 and to g
u = g - g.mean()
best_pert = -np.inf
for _ in range(20000):
    v = rng.standard_normal(4)
    v -= v.mean()
    v -= (v @ u) / (u @ u) * u
    if np.all(v >= 0):
        continue
    t = 0.5 * float(np.min(q[v < 0] / (-v[v < 0])))
    cand = q + t * v
    best_pert = max(best_pert, H_nats(cand))
print(f"\nbest exactly-feasible perturbation: H = {best_pert:.9f} nats"
      f"   (deficit {H_nats(q) - best_pert:.3e})")
assert abs(mean_at(beta) - mu) < 1e-9
assert abs(H_nats(q) - (beta * mu + np.log(w.sum()))) < 1e-9
assert best_pert < H_nats(q)

beta      = 0.13847146162892188
q         = [0.3082 0.2683 0.2336 0.1898]
E_q[g]    = 1.4  target 1.4
H(q)      = 1.37088248666545 nats
beta*mu + ln Z = 1.3708824866654503 nats

best exactly-feasible perturbation: H = 1.324113294 nats   (deficit 4.677e-02)
